<a href="https://colab.research.google.com/github/cwilson-su/Large-Scale-Data-Analytics-Labs/blob/main/LAB1_2026_chaines_de_traitement_ENONCE.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# LSDA 2026 — TP1 : préparation de données et chaînes de traitement Spark

Durée : 2 h — Outil : Spark SQL / pyspark sur Colab

---

## Ce qu'on travaille ici

Ce TP ne porte pas sur « savoir écrire une requête SQL ». Un assistant IA le fait mieux et
plus vite que vous, et ce n'est pas le sujet du cours. Ce qu'on travaille, c'est :

1. **construire une chaîne de traitement** : décomposer un besoin en un graphe de requêtes ;
2. **distinguer définir et exécuter** : savoir, avant d'appuyer sur Entrée, ce qui va réellement
   être calculé ;
3. **décider quoi matérialiser** (`cache`) et le **justifier par une mesure**, pas par intuition ;
4. **anticiper le comportement quand le volume de données augmente**.

## Ce qui est évalué

Votre **journal de bord** : les cellules marquées `[À COMPLÉTER]`. On y attend vos prédictions,
vos mesures de temps et vos interprétations. Une chaîne qui marche mais dont vous ne savez pas
expliquer le comportement ne vaut rien ici ; une chaîne imparfaite dont vous avez diagnostiqué
les défauts avec des mesures vaut beaucoup.

## Usage d'un assistant IA

**Autorisé et attendu.** En contrepartie, il est *déclaré* : la partie E vous demande vos prompts
et votre critique des réponses obtenues. Attention : l'assistant ne connaît pas *vos* mesures,
ni la machine sur laquelle vous tournez. Les questions de ce TP portent sur ce que produit
**votre** session.

## Déroulé indicatif

| | Partie | Durée |
|---|---|---|
| A | Définir n'est pas exécuter | 20 min |
| B | Mission : chaîne de traitement mobilité | 45 min |
| C | Diagnostiquer et optimiser une chaîne fournie | 30 min |
| D | Passage à l'échelle | 25 min |
| E | Synthèse et déclaration d'usage de l'IA | 10 min |

Les annexes (DuckDB, Geonames) sont pour aller plus loin, hors temps de séance.

# 0. Mise en route

Tout ce qui suit est **fourni** : exécutez les cellules dans l'ordre sans les modifier.
Aucun enjeu pédagogique ici, on veut juste une session Spark et des outils de mesure.

Vérifiez d'abord que des ressources de calcul sont allouées au notebook (RAM / disque en haut
à droite). Sinon, cliquez sur *Connecter*.

### 0.1 Imports et installation

In [ ]:
import os
import glob
import time
import datetime
from contextlib import contextmanager

In [ ]:
try:
    import pyspark
except ImportError:
    !pip install -q pyspark
    import pyspark

try:
    import findspark
except ImportError:
    !pip install -q findspark
    import findspark

try:
    from itables import init_notebook_mode
except ImportError:
    !pip -q install itables
    from itables import init_notebook_mode

init_notebook_mode(all_interactive=True)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 26.9 MB/s eta 0:00:00


### 0.2 Session Spark

In [ ]:
pyspark_dirs = ("/usr/local/lib/python*/dist-packages/pyspark",
                "/opt/tljh/user/envs/pyspark/lib/python3*/site-packages/pyspark",
                os.environ["HOME"] + "/.local/lib/python*/site-packages/pyspark")

existing_pyspark_dirs = [l[0] for l in [glob.glob(d) for d in pyspark_dirs] if len(l) > 0]
os.environ["SPARK_HOME"] = existing_pyspark_dirs[0]

java_dirs = ("/usr", "/opt/tljh/user/envs/pyspark")
os.environ["JAVA_HOME"] = [d for d in java_dirs if os.path.exists(d + "/bin/java")][0]

print("SPARK_HOME =", os.environ["SPARK_HOME"])
print("JAVA_HOME  =", os.environ["JAVA_HOME"])

SPARK_HOME = /usr/local/lib/python3.13/dist-packages/pyspark
JAVA_HOME  = /usr


In [ ]:
from pyspark.sql import SparkSession, Row
from pyspark import SparkConf
from pyspark.sql import *
from pyspark.sql.functions import *
from pyspark.sql.types import *

findspark.init()

def demarrer_spark():
    configLocale = SparkConf().setAppName("TP1").setMaster("local[*]") \
        .set("spark.executor.memory", "6G") \
        .set("spark.driver.memory", "6G") \
        .set("spark.sql.catalogImplementation", "in-memory")

    spark = SparkSession.builder.config(conf=configLocale).getOrCreate()
    spark.sparkContext.setLogLevel("ERROR")

    # on neutralise deux optimisations automatiques pour que les mesures soient lisibles
    spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "-1")
    # on ajuste le parallélisme à la taille de la VM colab (4 coeurs)
    spark.conf.set("spark.sql.shuffle.partitions", "4")
    # fuseau horaire fixé, pour que les mesures soient comparables d'une machine à l'autre
    spark.conf.set("spark.sql.session.timeZone", "UTC")

    print("session démarrée, id =", spark.sparkContext.applicationId)
    return spark

spark = demarrer_spark()

session démarrée, id = local-1790262256362


### 0.3 Affichage et cellules SQL

`display(requete)` affiche les premières lignes d'un résultat sous forme de tableau.
Le tag `%%sql` en première ligne d'une cellule permet d'y écrire directement du SQL :
plusieurs instructions peuvent être séparées par `;`, seule la dernière est affichée.

In [ ]:
def display(requete, n=30):
    """Affiche les n premières lignes du résultat d'une requête."""
    return requete.limit(n).toPandas()

In [ ]:
from IPython.core.magic import register_line_cell_magic

def _sans_commentaires(query):
    return "\n".join(l for l in query.split("\n") if not l.strip().startswith("--"))

@register_line_cell_magic
def sql(line, cell=None):
    """Exécute une ou plusieurs instructions SQL. Usage : %%sql"""
    val = cell if cell is not None else line
    requetes = [r.strip() for r in _sans_commentaires(val).split(";") if len(r.strip()) > 2]
    derniere, est_requete = None, False
    for r in requetes:
        derniere = spark.sql(r)
        est_requete = r.lower().startswith("select") or r.lower().startswith("with")
    return display(derniere) if est_requete else print("ok")

### 0.4 Téléchargement des données

In [ ]:
from urllib import request
import zipfile

PUBLIC_DATASET_URL = "https://gitlab.lip6.fr/naacke/tp/-/raw/main/dataset"
print("Les datasets disponibles :", PUBLIC_DATASET_URL.replace("/raw/", "/tree/"))

local_dir = "/local/data"
os.makedirs(local_dir, exist_ok=True)

def download_file(url, local_path):
    if not os.path.exists(local_path):
        print("téléchargement de", url)
        request.urlretrieve(url, local_path)
    else:
        print(local_path, "déjà présent")

def unzip_file(local_dir, file):
    with zipfile.ZipFile(os.path.join(local_dir, file), "r") as zip_ref:
        for f in zip_ref.namelist():
            if not os.path.exists(os.path.join(local_dir, f)):
                zip_ref.extract(f, local_dir)
                print("extrait :", f)

Les datasets disponibles : https://gitlab.lip6.fr/naacke/tp/-/tree/main/dataset


In [ ]:
# Données de mobilité issues du dataset YFCC (photos géolocalisées, région de Toronto).
# Un tuple = un utilisateur a vu un point d'intérêt (POI) à une date, pendant une séquence de visites.

for f in ["userVisits-Toro.csv", "POI-Toro.csv"]:
    download_file(os.path.join(PUBLIC_DATASET_URL, "YFCC-POI", f), os.path.join(local_dir, f))

os.listdir(local_dir)

téléchargement de https://gitlab.lip6.fr/naacke/tp/-/raw/main/dataset/YFCC-POI/userVisits-Toro.csv
téléchargement de https://gitlab.lip6.fr/naacke/tp/-/raw/main/dataset/YFCC-POI/POI-Toro.csv


['POI-Toro.csv', 'userVisits-Toro.csv']

### 0.5 Outils de mesure — **lisez cette cellule, vous allez l'utiliser partout**

Trois fonctions :

* `executer(r)` : produit **la totalité** du résultat de `r` sans l'afficher. C'est ce qui force
  réellement le calcul complet d'une requête. (`display(r)` n'en calcule qu'un extrait.)
  `r` peut être un DataFrame, un nom de vue, ou une requête SQL.
* `mesurer(r, repetitions=3)` : exécute plusieurs fois et renvoie les durées en secondes.
  **La première exécution n'est presque jamais comparable aux suivantes** : c'est justement
  une des choses à comprendre dans ce TP.
* `vider_cache()` : oublie tout ce qui a été matérialisé. À appeler avant une mesure « à froid ».

In [ ]:
import builtins   # attention : from pyspark.sql.functions import * masque min, max, sum, abs

@contextmanager
def chrono(libelle=""):
    t0 = time.perf_counter()
    yield
    print("%-45s %6.2f s" % (libelle, time.perf_counter() - t0))


def executer(r):
    """Produit tout le résultat de r, sans rien afficher ni rien garder."""
    if isinstance(r, str):
        r = spark.sql(r) if " " in r.strip() else spark.table(r)
    r.write.format("noop").mode("overwrite").save()


def mesurer(r, repetitions=3, libelle=""):
    durees = []
    for i in range(repetitions):
        t0 = time.perf_counter()
        executer(r)
        durees.append(time.perf_counter() - t0)
    print("%-45s %s   (min %.2f s)" % (
        libelle, " | ".join("%.2f" % d for d in durees), builtins.min(durees)))
    return durees


def vider_cache():
    spark.catalog.clearCache()
    print("cache vidé")


def nb_lignes(r):
    if isinstance(r, str):
        r = spark.sql(r) if " " in r.strip() else spark.table(r)
    return r.count()

print("outils de mesure prêts")

outils de mesure prêts


---
# Partie A — Définir n'est pas exécuter  *(≈ 20 min)*

## Protocole, à appliquer dans toute cette partie

Pour **chaque** cellule de mesure :

1. **Prédire d'abord.** Avant d'exécuter, écrivez dans la cellule `[À COMPLÉTER]` ce que vous
   attendez : est-ce que le fichier va être lu ? est-ce que ça va prendre du temps ?
2. **Mesurer ensuite.**
3. **Expliquer l'écart** entre votre prédiction et la mesure. C'est la partie qui compte.

Ne sautez pas l'étape 1 : une prédiction fausse écrite puis corrigée vaut mieux qu'une
prédiction juste devinée après coup.

## A.1 — Le fichier des visites

Regardons d'abord à quoi ressemble le fichier brut.

In [ ]:
chemin_visites = os.path.join(local_dir, "userVisits-Toro.csv")

with open(chemin_visites) as f:
    for i in range(3):
        print(f.readline().rstrip())

print("\nTaille du fichier : %.1f Mo" % (os.path.getsize(chemin_visites) / 1e6))
print("Nombre de lignes  :", builtins.sum(1 for _ in open(chemin_visites)))

"photoID";"userID";"dateTaken";"poiID";"poiTheme";"poiFreq";"seqID"
7941504100;"10007579@N00";1346844688;30;"Structure";1538;1
4886005532;"10012675@N05";1142731848;6;"Cultural";986;2

Taille du fichier : 2.3 Mo
Nombre de lignes  : 39420


Le fichier a-t-il une ligne d'en-tête ? Quel caractère sépare deux valeurs ?

**[À COMPLÉTER — A.1]**

*Votre réponse :*

## A.2 — Deux façons de lire le même fichier

Spark peut **inférer** le schéma (il devine les types) ou recevoir un **schéma explicite**.

**Prédiction :** laquelle des deux cellules suivantes va prendre du temps, et pourquoi ?
Écrivez votre réponse avant d'exécuter.

**[À COMPLÉTER — prédiction A.2]**

*Ma prédiction :*

In [ ]:
# Lecture n°1 : schéma inféré
with chrono("lecture avec inférence de schéma"):
    v_infere = (spark.read.format("csv")
                .option("header", "True").option("delimiter", ";")
                .option("inferSchema", "True")
                .load(chemin_visites))

v_infere.printSchema()

lecture avec inférence de schéma                6.33 s
root
 |-- photoID: long (nullable = true)
 |-- userID: string (nullable = true)
 |-- dateTaken: integer (nullable = true)
 |-- poiID: integer (nullable = true)
 |-- poiTheme: string (nullable = true)
 |-- poiFreq: integer (nullable = true)
 |-- seqID: integer (nullable = true)



In [ ]:
# Lecture n°2 : schéma explicite.
# Colonnes du fichier : photoID, userID, dateTaken, poiID, poiTheme, poiFreq, seqID
user_visits_schema = "..."   # <-- à compléter

with chrono("lecture avec schéma explicite"):
    user_visits = (spark.read.format("csv")
                   .option("header", "True").option("delimiter", ";")
                   .load(chemin_visites, schema=user_visits_schema))

user_visits.printSchema()

**[À COMPLÉTER — interprétation A.2]**

Les deux durées mesurées : ... et ...

Pourquoi cet écart ? Qu'est-ce que cela vous apprend sur le moment où un fichier est lu ?

*Votre réponse :*

## A.3 — `cache` : quand la matérialisation a-t-elle lieu ?

On enregistre la table dans le catalogue, puis on demande sa mise en cache.

**Prédiction :** la cellule `cache table` ci-dessous va-t-elle prendre du temps ?
Après elle, les données sont-elles en mémoire ?

**[À COMPLÉTER — prédiction A.3]**

*Ma prédiction :*

In [ ]:
user_visits.createOrReplaceTempView("user_visits")
vider_cache()

with chrono("cache table user_visits"):
    spark.sql("cache table user_visits")

print("is_cached :", spark.table("user_visits").is_cached)

## A.4 — Trois façons de « regarder » la même table

**Prédiction :** classez ces quatre opérations de la plus rapide à la plus lente,
et dites lesquelles lisent la totalité du fichier.

1. `display(user_visits, 3)`
2. `executer(user_visits)` — 1ʳᵉ fois
3. `executer(user_visits)` — 2ᵉ fois
4. `nb_lignes(user_visits)`

**[À COMPLÉTER — prédiction A.4]**

*Mon classement et ma justification :*

In [ ]:
with chrono("display 3 lignes"):
    r = display(user_visits, 3)

d = mesurer(user_visits, repetitions=3, libelle="executer (3 fois de suite)")

with chrono("nb_lignes"):
    n = nb_lignes(user_visits)
print("nombre de lignes :", n)

**[À COMPLÉTER — journal de mesures A]**

| opération | durée (s) | lit tout le fichier ? |
|---|---|---|
| lecture, schéma inféré | | |
| lecture, schéma explicite | | |
| `cache table` | | |
| `display(..., 3)` | | |
| `executer` 1ʳᵉ fois | | |
| `executer` 2ᵉ et 3ᵉ fois | | |

**Trois questions de synthèse :**

1. À quel moment exact le fichier a-t-il été lu pour la première fois ? Justifiez avec vos chiffres.
2. L'instruction `cache table` n'a rien coûté. Pourtant la 2ᵉ exécution est plus rapide que la 1ʳᵉ.
   Expliquez le mécanisme.
3. Si vous relancez `vider_cache()` puis `executer(user_visits)`, que prédisez-vous ?
   Vérifiez-le.

*Vos réponses :*

In [ ]:
# cellule libre pour vérifier la question 3

---
# Partie B — Mission : chaîne de traitement mobilité  *(≈ 45 min)*

On vous donne un **cahier des charges**, pas une marche à suivre. À vous de décomposer le
problème en une chaîne de requêtes.

## Les données

* `user_visits(photoID, userID, dateTaken, poiID, poiTheme, poiFreq, seqID)` — déjà chargée.
  Une ligne = une photo prise par un utilisateur, à une date (format Unix), sur un point
  d'intérêt (POI), pendant une séquence de visites `seqID`.
* `POI(poiID, poiName, latitude, longitude, theme)` — à charger (cellule ci-dessous).

## Les trois tables à produire

**B.1 — `Trajectoire(userID, seqID, listePOI)`**
`listePOI` est la liste des POI visités pendant la séquence, **dans l'ordre chronologique**,
sans répétition de deux POI consécutifs identiques.
*(Témoin : pour `seqID = 687`, `listePOI` vaut `[7, 16, 4, 8, 4, 16]`.)*

**B.2 — `Transition_relative(poi1, poi2, p)`**
Deux POI qui se suivent dans une trajectoire forment une *transition*. `p` est la part des
transitions partant de `poi1` qui aboutissent à `poi2`. Une séquence d'un seul POI ne produit
aucune transition.

**B.3 — `DureeMoyenneVisitePOI(poiID, duree_moyenne_min)`**
Durée moyenne de visite d'un POI, en minutes. Pour une suite d'événements consécutifs sur un
même POI dans une séquence, la durée de visite est l'écart entre le premier et le dernier
événement. Les visites de durée nulle ne comptent pas.

## Les contraintes

* Vous avez droit à **au plus deux vues matérialisées** (`cache table`) dans toute votre chaîne.
  Chacune devra être justifiée en B.5.
* Avant d'écrire la moindre requête, **dessinez votre chaîne** (partie B.0). C'est le vrai
  travail de cette partie.

Vous pouvez utiliser du SQL, l'API DataFrame, et des UDF Python si vous le jugez utile.

In [ ]:
# chargement de la table POI
poi_schema = "poiID long, poiName string, latitude double, longitude double, theme string"
poi = (spark.read.format("csv").option("header", "True").option("delimiter", ";")
       .load(os.path.join(local_dir, "POI-Toro.csv"), schema=poi_schema))
poi.createOrReplaceTempView("POI")
display(poi, 5)

## B.0 — Votre chaîne, avant d'écrire du code  **[À COMPLÉTER]**

Décrivez le graphe de vos requêtes : quelles vues intermédiaires, quelle vue dépend de quelle
autre, laquelle est lue plusieurs fois. Un schéma ASCII suffit, par exemple :

```
user_visits ──> V1 ──> V2 ──┬──> Transition_relative
                            └──> ...
```

*Votre chaîne :*



*Quelle(s) vue(s) sont lues par plusieurs autres ? (retenez-le pour B.5)*

## B.1 — `Trajectoire`

In [ ]:
# À vous.

**Vérification** (ne révèle pas la solution, dit seulement si le résultat est correct) :

In [ ]:
def verifier_trajectoire(nom="Trajectoire"):
    try:
        df = spark.table(nom)
    except Exception:
        return print("KO : la vue", nom, "n'existe pas")
    cols = [c.lower() for c in df.columns]
    manquantes = {c for c in ["userid", "seqid", "listepoi"] if c not in cols}
    if manquantes:
        return print("KO : colonnes manquantes :", manquantes)

    attendu_nb = spark.sql("select count(distinct seqID) as n from user_visits").first()["n"]
    obtenu_nb = df.count()
    ok = True
    if obtenu_nb != attendu_nb:
        print("KO : %d lignes, attendu %d (une ligne par séquence)" % (obtenu_nb, attendu_nb))
        ok = False

    cliste = df.columns[cols.index("listepoi")]
    r = df.where("seqID = 687").collect()
    if len(r) != 1:
        print("KO : la séquence 687 devrait apparaître une et une seule fois")
        ok = False
    else:
        liste = list(r[0][cliste])
        if liste != [7, 16, 4, 8, 4, 16]:
            print("KO : pour seqID=687 on obtient", liste, "au lieu de [7, 16, 4, 8, 4, 16]")
            ok = False

    # une trajectoire ne doit jamais contenir deux POI consécutifs identiques
    doublons = df.where(
        "size({c}) > 1 and exists(transform(sequence(1, size({c})-1), i -> {c}[i] = {c}[i-1]), b -> b)"
        .format(c=cliste)).count()
    if doublons > 0:
        print("KO : %d trajectoires contiennent deux POI consécutifs identiques" % doublons)
        ok = False

    if ok:
        print("OK : Trajectoire est conforme (%d séquences)" % obtenu_nb)

verifier_trajectoire()

## B.2 — `Transition_relative`

In [ ]:
# À vous.

In [ ]:
def verifier_transitions(nom="Transition_relative"):
    try:
        df = spark.table(nom)
    except Exception:
        return print("KO : la vue", nom, "n'existe pas")
    cols = [c.lower() for c in df.columns]
    if not {"poi1", "poi2", "p"} <= set(cols):
        return print("KO : colonnes attendues poi1, poi2, p ; obtenu :", df.columns)
    ok = True

    # invariant 1 : pour chaque poi1, les p somment à 1
    mauvais = (df.groupBy(df.columns[cols.index("poi1")])
                 .agg(sum(col(df.columns[cols.index("p")])).alias("s"))
                 .where("abs(s - 1) > 1e-9").count())
    if mauvais > 0:
        print("KO : pour %d POI de départ, la somme des p ne vaut pas 1" % mauvais)
        ok = False

    # invariant 2 : nb total de transitions = somme sur les séquences de (taille - 1)
    try:
        attendu = spark.sql("""select sum(case when size(listePOI) > 1
                                              then size(listePOI) - 1 else 0 end) as n
                               from Trajectoire""").first()["n"]
        print("   (contrôle : %d transitions attendues au total d'après Trajectoire)" % attendu)
    except Exception:
        pass

    if ok:
        print("OK : Transition_relative respecte les invariants (%d couples)" % df.count())

verifier_transitions()

## B.3 — `DureeMoyenneVisitePOI`

In [ ]:
# À vous.

In [ ]:
def verifier_duree(nom="DureeMoyenneVisitePOI"):
    try:
        df = spark.table(nom)
    except Exception:
        return print("KO : la vue", nom, "n'existe pas")
    cols = [c.lower() for c in df.columns]
    if not {"poiid"} <= set(cols) or len(df.columns) < 2:
        return print("KO : colonnes attendues poiID, duree_moyenne_min ; obtenu :", df.columns)
    cid = df.columns[cols.index("poiid")]
    cdur = [c for c in df.columns if c != cid][0]
    ok = True

    if df.where(col(cdur) <= 0).count() > 0:
        print("KO : certaines durées moyennes sont nulles ou négatives")
        ok = False
    if df.groupBy(cid).count().where("count > 1").count() > 0:
        print("KO : un POI apparaît plusieurs fois")
        ok = False
    inconnus = df.join(spark.table("POI"), col(cid) == col("poiID"), "left_anti").count()
    if inconnus > 0:
        print("KO : %d identifiants ne correspondent à aucun POI" % inconnus)
        ok = False
    if ok:
        print("OK : DureeMoyenneVisitePOI est plausible (%d POI, durée moyenne globale %.1f min)"
              % (df.count(), df.agg(avg(cdur)).first()[0]))

verifier_duree()

## B.4 — Mesurer votre chaîne  **[À COMPLÉTER]**

Mesurez le coût de votre chaîne complète, d'abord **sans aucune matérialisation**,
puis avec les matérialisations que vous avez choisies (rappel : deux au maximum).

In [ ]:
vider_cache()
print("--- sans matérialisation ---")
mesurer("Transition_relative", 2, "Transition_relative")
mesurer("DureeMoyenneVisitePOI", 2, "DureeMoyenneVisitePOI")

# ... placez ici vos cache table, puis remesurez

## B.5 — Justification  **[À COMPLÉTER]**

1. Quelles vues avez-vous matérialisées, et pourquoi **celles-là** ? Appuyez-vous sur votre
   graphe de B.0 (combien de fois chaque vue est-elle lue ?) et sur vos mesures de B.4.
2. Y a-t-il une vue de votre chaîne qu'il serait **inutile** de matérialiser même si on vous
   en donnait le droit ? Laquelle et pourquoi ?
3. Dans quel ordre avez-vous mis les `cache table` par rapport aux requêtes ? Est-ce que l'ordre
   change quelque chose ? (testez si vous hésitez)

*Vos réponses :*

---
# Partie C — Diagnostiquer et optimiser une chaîne fournie  *(≈ 30 min)*

On vous donne une chaîne qui **marche** : elle calcule, pour chaque thème de POI, l'**heure de
la journée où ce thème est le plus photographié**. Elle est écrite comme on écrit quand on ne
réfléchit pas au moteur d'exécution.

Votre travail : mesurer, diagnostiquer, réécrire, **prouver le gain**.

La chaîne est donnée sous forme de fonction, pour pouvoir la relancer sur d'autres données
en partie D.

In [ ]:
# ---------------------------------------------------------------
# CHAÎNE FOURNIE — ne la modifiez pas, écrivez votre version à côté
# ---------------------------------------------------------------

def heure_python(ts):
    from datetime import datetime, timezone
    return datetime.fromtimestamp(ts, tz=timezone.utc).hour

spark.udf.register("heure_python", heure_python, IntegerType())


def chaine_lente(visites):
    visites.createOrReplaceTempView("V")

    spark.sql("create or replace temp view L1 as select * from V order by photoID")

    spark.sql("""create or replace temp view L2 as
                 select seqID, poiID, poiTheme, heure_python(dateTaken) as heure
                 from L1""")

    spark.sql("""create or replace temp view L3 as
                 select poiTheme, heure, count(*) as nb
                 from L2 group by poiTheme, heure""")

    spark.sql("""create or replace temp view L4 as
                 select poiTheme, max(nb) as nbmax
                 from L3 group by poiTheme""")

    spark.sql("""create or replace temp view L5 as
                 select L3.poiTheme, L3.heure, L3.nb
                 from L3 join L4 on L3.poiTheme = L4.poiTheme and L3.nb = L4.nbmax""")

    return spark.table("L5")

resultat_lent = chaine_lente(user_visits)
display(resultat_lent)

## C.1 — Mesure de référence  **[À COMPLÉTER]**

**Avant de mesurer**, lisez la chaîne et notez ici les endroits qui vous paraissent coûteux
inutilement. Combien y en a-t-il selon vous ?

*Ma prédiction :*

In [ ]:
vider_cache()
mesurer(resultat_lent, repetitions=3, libelle="chaîne fournie")

## C.2 — Diagnostic  **[À COMPLÉTER]**

La chaîne contient **trois** défauts de conception, chacun lié à une notion du cours.
Identifiez-les et, pour chacun, dites :

| | défaut | notion du cours concernée | correction envisagée |
|---|---|---|---|
| 1 | | | |
| 2 | | | |
| 3 | | | |

*Indice méthodologique, pas de solution* : pour chaque vue `L1`…`L5`, demandez-vous
(a) combien de fois elle est lue par la suite de la chaîne, (b) si le travail qu'elle fait sert
au résultat final, (c) où le calcul a lieu — dans le moteur Spark ou ailleurs.

## C.3 — Votre version

In [ ]:
def chaine_rapide(visites):
    # à vous
    pass

resultat_rapide = chaine_rapide(user_visits)
display(resultat_rapide)

## C.4 — Preuve du gain  **[À COMPLÉTER]**

Deux choses à établir : votre version est **plus rapide**, et elle donne **le même résultat**.

In [ ]:
# même résultat ?
a = chaine_lente(user_visits).toPandas().sort_values(["poiTheme", "heure"]).reset_index(drop=True)
b = chaine_rapide(user_visits).toPandas().sort_values(["poiTheme", "heure"]).reset_index(drop=True)
print("résultats identiques :", a.equals(b))

# plus rapide ?
vider_cache()
d_lent = mesurer(chaine_lente(user_visits), 3, "chaîne fournie")
vider_cache()
d_rapide = mesurer(chaine_rapide(user_visits), 3, "votre chaîne")
print("\ngain : x%.1f" % (builtins.min(d_lent) / builtins.min(d_rapide)))

**Vos conclusions :**

| version | durée min (s) | gain |
|---|---|---|
| chaîne fournie | | 1 |
| votre chaîne | | |

1. Le gain est-il à la hauteur de ce que vous attendiez ? Sur un jeu de 40 000 lignes, un
   écart de quelques dixièmes de seconde est-il une preuve solide ? Qu'est-ce qui pourrait
   fausser la mesure ?
2. Parmi vos trois corrections, laquelle rapporte le plus ? Vérifiez-le en n'en appliquant
   qu'une seule à la fois.

*Vos réponses :*

---
# Partie D — Passage à l'échelle  *(≈ 25 min)*

Le jeu de Toronto fait quelques dizaines de milliers de lignes : c'est du « petit » données.
Tout va vite, y compris ce qui est mal écrit. La question qui intéresse ce cours est
ailleurs : **comment le coût évolue-t-il quand le nombre de lignes augmente ?**

On fabrique donc des versions plus grandes du jeu de données. `amplifier(k)` en produit une
copie k fois plus grosse : k jeux d'utilisateurs et de séquences distincts, décalés dans le
temps d'un nombre entier d'années. La structure de chaque séquence est **inchangée**, donc les
agrégats par POI et le profil horaire par thème doivent rester **identiques** — ce sera notre
contrôle de cohérence.

In [ ]:
def amplifier(k, base_dir="/local/data/visites_x"):
    """Fabrique (une seule fois) un jeu k fois plus gros. Renvoie le chemin."""
    chemin = base_dir + str(k)
    if os.path.exists(chemin):
        return chemin
    base = (spark.read.format("csv").option("header", "True").option("delimiter", ";")
            .load(chemin_visites, schema=user_visits_schema))
    copies = spark.range(k).withColumnRenamed("id", "copie")
    amplifie = (base.crossJoin(copies).select(
        (col("photoID") * 1000 + col("copie")).alias("photoID"),
        concat(col("userID"), lit("_"), col("copie")).alias("userID"),
        (col("dateTaken") + col("copie") * lit(31536000)).alias("dateTaken"),   # + k années
        col("poiID"), col("poiTheme"), col("poiFreq"),
        (col("seqID") * 1000 + col("copie")).alias("seqID")))
    (amplifie.write.mode("overwrite")
     .option("header", "True").option("delimiter", ";").csv(chemin))
    print("généré :", chemin)
    return chemin


def lire_amplifie(k):
    chemin = amplifier(k)
    return (spark.read.format("csv").option("header", "True").option("delimiter", ";")
            .load(chemin, schema=user_visits_schema))

# test
v2 = lire_amplifie(2)
print("x1 :", user_visits.count(), "lignes")
print("x2 :", v2.count(), "lignes")

## D.1 — Protocole  **[À COMPLÉTER]**

Vous allez mesurer les deux chaînes de la partie C pour plusieurs tailles.

**Prédisez d'abord.** Si le nombre de lignes est multiplié par 10 :

* la durée de la chaîne fournie est multipliée par... ?
* celle de votre chaîne optimisée par... ?
* l'écart entre les deux reste-t-il constant, se creuse-t-il, se réduit-il ? Pourquoi ?

*Mes prédictions :*

In [ ]:
n_base = user_visits.count()
tailles = [1, 2, 5, 10, 20]      # augmentez si vos durées restent trop petites (< 2 s)
mesures = []

for k in tailles:
    v = lire_amplifie(k)
    v.createOrReplaceTempView("V")

    vider_cache()
    t_lent = mesurer(chaine_lente(v), 1, "k=%2d  chaîne fournie" % k)[0]

    vider_cache()
    t_rapide = mesurer(chaine_rapide(v), 1, "k=%2d  chaîne optimisée" % k)[0]

    mesures.append({"k": k, "lignes": k * n_base, "lente": t_lent, "rapide": t_rapide})

import pandas as pd
tableau = pd.DataFrame(mesures)
tableau["gain"] = tableau["lente"] / tableau["rapide"]
tableau

In [ ]:
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
ax1.plot(tableau["k"], tableau["lente"], "o-", label="chaîne fournie")
ax1.plot(tableau["k"], tableau["rapide"], "s-", label="chaîne optimisée")
ax1.set_xlabel("facteur d'amplification k"); ax1.set_ylabel("durée (s)")
ax1.set_title("Coût en fonction du volume"); ax1.legend(); ax1.grid(alpha=.3)

ax2.plot(tableau["k"], tableau["gain"], "d-", color="firebrick")
ax2.set_xlabel("facteur d'amplification k"); ax2.set_ylabel("gain (x)")
ax2.set_title("Écart entre les deux chaînes"); ax2.grid(alpha=.3)
plt.tight_layout(); plt.show()

## D.2 — Contrôle de cohérence

Le profil horaire ne dépend pas du nombre de copies : le résultat doit être le même pour tout k.
Vérifiez-le — une optimisation qui change le résultat n'est pas une optimisation.

In [ ]:
r1 = chaine_rapide(lire_amplifie(1)).toPandas().sort_values("poiTheme").reset_index(drop=True)
r10 = chaine_rapide(lire_amplifie(10)).toPandas().sort_values("poiTheme").reset_index(drop=True)
print(r1[["poiTheme", "heure"]].equals(r10[["poiTheme", "heure"]]))
display_cols = ["poiTheme", "heure", "nb"]
print(r1[display_cols].to_string(index=False))
print(r10[display_cols].to_string(index=False))

## D.3 — Interprétation  **[À COMPLÉTER]**

1. Tracez vos deux courbes. Sont-elles linéaires en k ? Si non, dites en quoi elles s'en écartent
   et à partir de quel k.
2. Une courbe qui ne passe pas par l'origine, ça veut dire quoi ? Estimez le **coût fixe**
   (indépendant du volume) de chaque chaîne.
3. L'écart entre les deux chaînes évolue-t-il avec k ? Reliez ce que vous observez aux trois
   défauts diagnostiqués en C.2 : lequel coûte proportionnellement au volume, lequel non ?
4. Avec la pente que vous avez mesurée, combien de temps prendrait la chaîne fournie sur un
   fichier de 10 milliards de lignes ? Cette extrapolation est-elle raisonnable ? Qu'est-ce
   qui, dans votre configuration, finirait par la rendre fausse ?
5. Mesurez maintenant la **deuxième** exécution de la chaîne optimisée à k = 20, après avoir
   matérialisé ce qui vous semble pertinent. Le bénéfice du cache grandit-il avec le volume ?

*Vos réponses :*

In [ ]:
# cellule libre pour la question 5

---
# Partie E — Synthèse  *(≈ 10 min)*

**[À COMPLÉTER — à rendre]**

1. En vous appuyant **sur vos mesures**, donnez la règle que vous appliqueriez pour décider de
   matérialiser une vue dans une chaîne. Trois lignes maximum, et pas de « ça dépend ».
2. Citez une chose que vous croyiez vraie en arrivant et que vos mesures ont contredite.
3. Dans votre chaîne de la partie B, quelle vue matérialiseriez-vous si le jeu de données faisait
   100 fois la taille actuelle ? Est-ce la même réponse qu'en B.5 ? Pourquoi ?
4. Le cours dit qu'une requête n'est exécutée qu'« à la demande ». Donnez deux exemples tirés
   de *votre* séance où cela vous a induit en erreur dans une mesure.

*Vos réponses :*

## Déclaration d'usage d'un assistant IA  **[À COMPLÉTER — à rendre]**

1. Pour quelles parties du TP avez-vous utilisé un assistant ? Recopiez **un** de vos prompts.
2. Sa réponse était-elle directement correcte ? Si oui, qu'avez-vous dû comprendre par vous-même
   pour savoir qu'elle l'était ?
3. Citez **une** réponse de l'assistant que vos mesures ont invalidée ou nuancée
   (par exemple sur l'endroit où placer un cache, ou sur l'intérêt d'une UDF). Si vous n'en avez
   trouvé aucune, posez-lui la question 3 de la partie E et confrontez sa réponse à vos chiffres.

*Vos réponses :*

---
# Annexe 1 — Pour aller plus loin : les mêmes traitements avec DuckDB

DuckDB est un moteur SQL analytique **mono-machine**, sans cluster, sans exécution distribuée.
Sur des volumes qui tiennent sur une machine, il est souvent beaucoup plus rapide que Spark.
L'exercice : refaire la partie C avec DuckDB et comparer les courbes de la partie D.

La question intéressante n'est pas « lequel est le meilleur » mais **à partir de quel volume
l'ordre s'inverse-t-il**, et pourquoi.

In [ ]:
try:
    import duckdb
except ImportError:
    !pip install -q duckdb
    import duckdb

db = duckdb.connect(":memory:")

# DuckDB lit directement le CSV
db.sql("""create or replace view visites as
          select * from read_csv('/local/data/userVisits-Toro.csv', delim=';', header=true)""")

db.sql("""select poiTheme, extract(hour from to_timestamp(dateTaken)) as heure, count(*) as nb
          from visites group by 1, 2 order by nb desc limit 10""").df()

In [ ]:
# à vous : reprendre la chaîne optimisée de la partie C en DuckDB,
# et refaire les mesures de la partie D sur les mêmes fichiers amplifiés.

# Annexe 2 — Pour aller plus loin : croiser avec Geonames

Geonames est une base ouverte de points géographiques du monde entier (plusieurs millions de
lignes, contrairement à nos 40 000). De quoi refaire les mesures de la partie D sur des données
réelles plutôt que dupliquées.

Le schéma est décrit dans le
[readme.txt](https://gitlab.lip6.fr/naacke/tp/-/blob/main/dataset/geonames/readme.txt)
du dossier [geonames](https://gitlab.lip6.fr/naacke/tp/-/tree/main/dataset/geonames).

In [ ]:
# chargement (environ 40 s)
download_file(os.path.join(PUBLIC_DATASET_URL, "geonames", "allCountries.zip"),
              os.path.join(local_dir, "allCountries.zip"))
unzip_file(local_dir, "allCountries.zip")

geonames = (spark.read.format("csv").option("header", "False").option("delimiter", "\t")
            .load(os.path.join(local_dir, "allCountries.txt")))
geonames.createOrReplaceTempView("geonames")
display(geonames, 3)

**a)** Définir une vue `Geonames2` avec un schéma nommé et typé, limité à : identifiant
(`geonameID`), noms (`name`, `alternate_name`), position (`latitude`, `longitude`),
catégorie (`feature_class`, `feature_code`), pays (`country_code`).

**b)** En déduire `Geonames_canada`, restreinte au Canada. Combien de lignes ?

**c)** Associer chaque POI de Toronto à son entrée Geonames, pour l'enrichir de son
`feature_code`. Le rapprochement se fait sur la position géographique : à vous de définir ce que
« correspondre » veut dire et de justifier votre critère.

**d)** Mesurer. Cette jointure est-elle coûteuse ? Que se passe-t-il si vous filtrez sur le
Canada **avant** la jointure plutôt qu'après ? Comparez, et concluez.

In [ ]:
# à vous